# Diagnostics in ProbPipe

# This notebook demonstrates the current diagnostics design in ProbPipe.
#
# Diagnostics use a single public layer of **in-place writers** that
# compute diagnostics and write results into ``posterior._auxiliary``:
#
#     add_mcmc_diagnostics(posterior)
#     add_ppc(posterior, ...)
#     add_loo(posterior, ...)
#
# Diagnostics in ProbPipe use a two-tree layout:
#
#     posterior._auxiliary
#     ├── /arviz/          # ArviZ-compatible DataTree
#     └── /diagnostics/    # ProbPipe diagnostic summaries

## 1. Setup

In [2]:
import warnings
warnings.filterwarnings("ignore", message=r"Explicitly requested dtype.*float64.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import tensorflow_probability.substrates.jax.glm as tfp_glm

import probpipe.diagnostics as diagnostics
importlib.reload(diagnostics)

from probpipe.diagnostics import (
    add_mcmc_diagnostics,
    add_rhat,
    add_ess,
    add_mcse,
    add_ppc,
    # add_loo,
)

from probpipe import (
    Record,
    NumericRecordArray,
    NumericRecordTemplate,
    Normal,
    ProductDistribution,
    EmpiricalDistribution,
    BootstrapReplicateDistribution,
    GLMLikelihood,
    SimpleModel,
    condition_on,
    mean,
    variance,
    workflow_function,
    provenance_ancestors,
)

from probpipe.modeling import IncrementalConditioner
from probpipe.custom_types import ArrayLike

In [3]:
# Helper: inspect _auxiliary layout

def has_path(tree, path):
    """Check whether a slash-separated path exists in a DataTree-like object."""
    if tree is None:
        return False

    node = tree
    for part in path.strip("/").split("/"):
        if not part:
            continue
        try:
            node = node[part]
        except Exception:
            return False

    return True


def show_auxiliary_paths(posterior):
    """Print expected diagnostics and ArviZ paths."""
    aux = getattr(posterior, "_auxiliary", None)

    print("posterior._auxiliary is None:", aux is None)

    paths = [
        "arviz",
        "arviz/posterior",
        "arviz/sample_stats",
        "arviz/observed_data",
        "arviz/posterior_predictive",
        "arviz/log_likelihood",
        "diagnostics",
        "diagnostics/mcmc",
        "diagnostics/runs",
        "diagnostics/runs/ppc",
        "diagnostics/runs/loo",
        "diagnostics/runs/spc",
    ]

    for path in paths:
        print(f"{path:35s}", has_path(aux, path))

## 2. Data

We use the horseshoe crabs data. The response is the number of satellites, and the predictor is crab width.

In [4]:
# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_csv("../tutorials/data/horseshoe_crabs.csv")

df.head()
print(df)




     width_cm  satellites
0        28.3           8
1        22.5           0
2        26.0           9
3        24.8           0
4        26.0           4
..        ...         ...
167      32.5           0
168      23.5           0
169      26.5           6
170      25.5           3
171      22.5           1

[172 rows x 2 columns]


In [5]:
# Standardize width and assemble a NumericRecordArray with fields X and y.

@workflow_function
def prep_data(width: ArrayLike, satellites: ArrayLike) -> NumericRecordArray:
    width = np.asarray(width, dtype=np.float32)
    width_z = ((width - np.mean(width)) / np.std(width)).astype(np.float32)

    y = np.asarray(satellites, dtype=np.float32)

    template = NumericRecordTemplate(X=(), y=())

    return NumericRecordArray(
        {"X": width_z, "y": y},
        batch_shape=(len(width),),
        template=template,
    )


data = prep_data(df["width_cm"], df["satellites"])

print(data)
print("source:", data.source)

NumericRecordArray(batch_shape=(172,), X=array(shape=(172,)), y=array(shape=(172,)))
source: Provenance('workflow.prep_data', parents=[])


## 3. Poisson Regression Model

We fit a Poisson regression model for count data:

$$
y_i \sim \operatorname{Poisson}\left(\exp(\eta_i)\right),
\qquad
\eta_i = \alpha + x_i \beta.
$$

`GLMLikelihood` wraps a TFP GLM family and a design matrix into a ProbPipe likelihood.

In [6]:
lik_poisson = GLMLikelihood(tfp_glm.Poisson(), data["X"])

prior = ProductDistribution(
    Normal(loc=0.0, scale=jnp.sqrt(5.0), name="intercept"),
    Normal(loc=0.0, scale=jnp.sqrt(5.0), name="slope"),
)

model_poisson = SimpleModel(prior, lik_poisson, name="poisson")

posterior = condition_on(model_poisson, data["y"])

print(posterior)

ApproximateDistribution(algorithm='blackjax_nuts', num_chains=1, num_draws=1000, event_shapes={'intercept': (), 'slope': ()})


In [7]:
# Inspect auxiliary structure immediately after inference.
# Ideally, condition_on has populated the ArviZ-compatible subtree.

show_auxiliary_paths(posterior)

posterior._auxiliary is None: False
arviz                               True
arviz/posterior                     True
arviz/sample_stats                  True
arviz/observed_data                 False
arviz/posterior_predictive          False
arviz/log_likelihood                False
diagnostics                         False
diagnostics/mcmc                    False
diagnostics/runs                    False
diagnostics/runs/ppc                False
diagnostics/runs/loo                False
diagnostics/runs/spc                False


## 4. MCMC diagnostics
#
# Run diagnostics and write results into ``posterior._auxiliary``:
#
#     add_mcmc_diagnostics(posterior)   # all metrics at once
#     add_rhat(posterior)               # R-hat only
#     add_ess(posterior)                # ESS only
#     add_mcse(posterior)               # MCSE only
#
# Results are written under:
#     posterior._auxiliary["diagnostics"]["mcmc"]
#
# Read back via ``posterior.diagnostics`` (a DiagnosticsView):


In [8]:
add_mcmc_diagnostics(posterior)

mcmc = posterior.diagnostics.mcmc   # MCMCView

print("R-hat:     ", mcmc.rhat)
print("ESS bulk:  ", mcmc.ess_bulk)
print("ESS tail:  ", mcmc.ess_tail)
print("MCSE mean: ", mcmc.mcse_mean)
print()

if mcmc.warnings:
    for w in mcmc.warnings:
        print("⚠", w)
else:
    print("✓ All MCMC diagnostics passed.")

R-hat:      {'intercept': NotComputed('R-hat requires at least 2 chains'), 'slope': NotComputed('R-hat requires at least 2 chains')}
ESS bulk:   {'intercept': 433.10269057959124, 'slope': 410.05555168331085}
ESS tail:   {'intercept': 480.608713116952, 'slope': 516.254330481698}
MCSE mean:  {'intercept': 0.0028643168356568727, 'slope': 0.0023220882008709773}

✓ All MCMC diagnostics passed.


In [9]:
# Or use the built-in formatted table
print()
print(posterior.diagnostics.summary_table())


Parameter     R-hat     ESS bulk    ESS tail    MCSE mean   
------------------------------------------------------------
intercept     N/A       433.1027    480.6087    0.0029      
slope         N/A       410.0556    516.2543    0.0023      

✓  All available diagnostics passed.


## 5. Posterior predictive checks

``add_ppc`` runs one or more test statistics against replicated datasets
and writes results into ``posterior._auxiliary/diagnostics/runs/ppc/``.
Read back via ``posterior.diagnostics.ppc`` (a PPCView).

In [10]:
var_mean_ratio = lambda d: float(jnp.var(d) / jnp.maximum(jnp.mean(d), 1e-6))
var_mean_ratio.__name__ = "var_mean_ratio"

zero_fraction = lambda d: float(jnp.mean(d == 0))
zero_fraction.__name__ = "zero_fraction"

add_ppc(
    posterior,
    test_fns=[var_mean_ratio, zero_fraction],
    observed_data=data["y"],
    generative_likelihood=lik_poisson,
    n_replications=500,
)

ppc = posterior.diagnostics.ppc     # PPCView

print("p-values:  ", ppc.p_values)
print("observed:  ", ppc.observed)
print()

# Combined result dict
for name, res in ppc.result.items():
    print(f"  {name}: observed={res['observed']:.3f}, p={res['p_value']:.3f}")

p-values:   {'var_mean_ratio': 0.0, 'zero_fraction': 0.0}
observed:   {'var_mean_ratio': 4.140847682952881, 'zero_fraction': 0.4651162624359131}

  var_mean_ratio: observed=4.141, p=0.000
  zero_fraction: observed=0.465, p=0.000


## 6. LOO-PSIS

``add_loo`` computes PSIS-LOO and writes results into
``posterior._auxiliary/diagnostics/runs/loo/``.
Read back via ``posterior.diagnostics.loo`` (a LOOView).

In [11]:
add_loo(posterior)

loo = posterior.diagnostics.loo     # LOOView

print("ELPD-LOO: ", loo.elpd_loo)
print("SE:       ", loo.se)
print("LOO-IC:   ", loo.looic)
print("Pareto-k max: ", loo.pareto_k_max)
print()

if loo.warnings:
    for w in loo.warnings:
        print("⚠", w)
else:
    print("✓ LOO diagnostics passed.")

NameError: name 'add_loo' is not defined

### Summary

| | Poisson |
|---|---|
| MCMC convergence | ✓ |
| PPC var/mean ratio | ✗ |
| PPC zero fraction | ✗ |

The Poisson model converges but fails both predictive checks — it cannot
capture overdispersion. Use a negative binomial model for count data
with overdispersion.